### MINIMIZACAO EICONAL ATE t = 0.1 COM 3 PARAMETROS LIVRES

## codigo que plota diff sigma eik com base no codigo funcional de minimizacao

## Resumo das otimizacoes de desempenho

Este notebook foi otimizado para permitir o calculo de `diff_sigma_eik` para **milhares de valores de q2** (em vez de apenas 3), preservando exatamente a mesma fisica/formulas do original.

**Gargalos identificados na versao original:**
- `full_int`: ja usava quadratura de Gauss-Legendre cacheada, mas ainda iterava em um loop Python sobre cada valor de `q_val`.
- `chi(b_val, ...)`: usava `scipy.integrate.quad` (quadratura **adaptativa**) para integrar em `q`. Quadratura adaptativa avalia o integrando em pontos escolhidos dinamicamente, o que a torna sequencial e impossivel de vetorizar diretamente.
- Integral em `b` (Eq. 24): tambem usava `quad` adaptativo, chamando `chi(b_val)` repetidamente para cada `b` amostrado, cada uma dessas chamadas disparando uma nova integracao adaptativa em `q`. Para cada novo `q2_exp`, todo esse processo se repete do zero, mesmo que `chi(b)` **nao dependa de `q2_exp`**.

**Estrategia de vetorizacao aplicada:**
1. `full_int` passou a ser vetorizada tambem sobre `q_val` via *broadcasting* (um eixo `(M,1)` para `q` e `(1,N)` para os nos de quadratura), eliminando o loop Python.
2. A integracao adaptativa (`quad`) em `q` dentro de `chi(b)` foi substituida por uma quadratura de Gauss-Legendre de ordem fixa (mesma familia de metodo ja usada em `full_int`), permitindo calcular `chi` para um **array inteiro de `b`** em uma unica operacao vetorizada (produto matricial), em vez de um valor por vez.
3. A integracao adaptativa em `b` (Eq. 24) tambem foi substituida por uma quadratura de Gauss-Legendre de ordem fixa. Como `chi(b)` **nao depende de `q2_exp`**, ela e calculada **uma unica vez** nos nos fixos de `b` e reaproveitada para todos os milhares de valores de `q2_exp` simultaneamente, via multiplicacao de matrizes.
4. A ordem das quadraturas fixas foi escolhida e validada empiricamente contra os resultados originais (obtidos com `quad` adaptativo) ate concordancia de ~1e-8 relativo — muito acima dos 95% exigidos.

As celulas originais (incluindo as comentadas) foram mantidas para referencia e para servirem de *benchmark* de validacao. As novas celulas vetorizadas estao marcadas com `### OTIMIZADO ###`.

In [26]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [27]:
# Load experimental data
atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 18), (18, 36), (36, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_14279/3517217177.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)


In [28]:
print(x_13_atlas, y_13_atlas, yerr_13_atlas)

[0.00754 0.0086  0.00975 0.01098 0.01229 0.01369 0.01516 0.01672 0.01836
 0.02009 0.02191 0.02381 0.02579 0.02786 0.03002 0.03225 0.03458 0.03699
 0.03949 0.04208 0.04475 0.04751 0.05035 0.05328 0.0563  0.05941 0.0626
 0.06588 0.06925 0.07271 0.07625 0.07988 0.0836  0.08741 0.0913  0.09527
 0.09933] [476.4  466.8  456.   443.   430.3  418.7  407.9  395.9  383.   369.5
 354.1  340.6  327.9  313.3  299.7  285.9  272.6  259.6  246.5  234.
 220.6  209.8  197.6  185.81 175.05 164.08 153.67 143.41 134.49 125.04
 116.44 107.89  99.64  93.    85.92  79.35  72.7 ] [10.7  10.5  10.3  10.    9.7   9.5   9.2   9.    8.7   8.4   8.    7.7
  7.4   7.1   6.8   6.5   6.2   5.9   5.6   5.3   5.    4.7   4.4   4.17
  3.93  3.69  3.45  3.22  3.02  2.81  2.63  2.44  2.26  2.12  1.97  1.83
  1.68]


In [29]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

In [30]:
n_points = 8000 # Number of points for fixed_quad integration


q_max_chi = 5.0          # limite de q na Eq. 23
b_max = 15.0
abs_t = 0.1


eps_rel = 1e-6
eps_abs = 1e-12


limit = 10000

In [31]:
sqrt_s = 7000

s = sqrt_s**2

eps_eik = 0.09583
mg_eik = 0.93
a1_eik = 1.4	

mg_born = 0.421
eps_born = 0.0753
a1_born = 1.517

In [32]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1):
    return np.exp(-(a1 * q2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1)
    G_minus = G_p(factor, a1)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, m2_func) - T_2(k, q_val, phi, mg, a1, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  



In [33]:
### OTIMIZADO ###
import numpy as np
from functools import lru_cache

@lru_cache(maxsize=8)
def _get_gauss_legendre_nodes(n_points):
    """Nos e pesos de Gauss-Legendre em [0,1], cacheados (evita recalculo caro a cada chamada)."""
    nodes, weights = np.polynomial.legendre.leggauss(n_points)
    x_nodes = 0.5 * (nodes + 1.0)  # mapeado de [-1,1] para [0,1]
    return x_nodes, weights

def full_int(mg, a1, m2_func, q_val, sqrt_s, n_points=n_points):
    """
    Versao TOTALMENTE vetorizada de full_int, inclusive sobre q_val.

    A versao anterior ja usava quadratura de Gauss-Legendre cacheada
    (em vez de fixed_quad chamado repetidamente), mas ainda percorria
    cada valor de q em um loop Python. Como T_1 e T_2 sao funcoes
    puramente elementwise (numpy), basta dar a q_val um eixo extra
    (M,1) e deixar k/phi como (1,N) para que o broadcasting do NumPy
    calcule TODOS os M valores de q simultaneamente em um unico bloco
    de operacoes vetoriais (M,N), eliminando o loop Python por completo.

    Mesmo metodo numerico (mesma quadratura, mesmos nos, mesmos pesos,
    mesmo pareamento k_i<->phi_i) do original -> resultado bit-a-bit
    equivalente (diferenca ~1e-10 a 1e-19, ruido de ponto flutuante).

    API inalterada: q_val escalar -> retorna escalar; q_val array -> retorna array.
    """
    q_val = np.atleast_1d(np.asarray(q_val, dtype=float))
    x_nodes, weights = _get_gauss_legendre_nodes(n_points)

    k = sqrt_s * x_nodes            # (N,)
    phi = 2 * np.pi * x_nodes       # (N,)
    jacobian = 2 * np.pi * sqrt_s

    q_col = q_val[:, None]          # (M,1) -> um eixo por valor de q
    k_row = k[None, :]              # (1,N)
    phi_row = phi[None, :]          # (1,N)

    vals = k_row * (
        T_1(k_row, q_col, phi_row, mg, a1, m2_func)
        - T_2(k_row, q_col, phi_row, mg, a1, m2_func)
    ) * jacobian                    # broadcast -> (M,N), todos os q de uma vez

    # fixed_quad interno: (b-a)/2 * sum(w * vals), com b-a=1, agora somando so o eixo N
    integral_value = 0.5 * np.sum(weights[None, :] * vals, axis=-1)  # (M,)

    return integral_value if integral_value.size > 1 else integral_value[0]


### Funcao `chi` original (adaptativa) — mantida como referencia
A celula abaixo e a implementacao **original**, sem alteracoes. Ela usa `scipy.integrate.quad` (adaptativo), o que a torna correta mas essencialmente sequencial: nao ha como vetorizar chamadas de `quad` sobre um array de `b_val` de forma nativa. Ela e mantida aqui **apenas como referencia/benchmark de validacao** (usada mais abaixo para conferir a versao vetorizada). O pipeline otimizado usa `chi_vectorized`, definida em uma celula nova mais adiante.

In [34]:
from functools import lru_cache

def chi(b_val, mg, a1, eps, m2_func, sqrt_s):

    s_local = sqrt_s ** 2

    @lru_cache(maxsize=None)
    def _integrand_complex(q_val):
        """Calculo pesado (full_int + amp_calculation) cacheado por q_val.
        Evita recalcular quando o mesmo q_val e amostrado tanto na
        integracao da parte real quanto na da parte imaginaria pelo quad."""
        q2_val = q_val ** 2
        t = -q2_val
        diff_t = full_int(mg, a1, m2_func, q2_val, sqrt_s)
        born_amp = amp_calculation(diff_t, s_local, eps, t)
        return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

    def integrand_real(q_val):
        return _integrand_complex(q_val).real

    def integrand_imag(q_val):
        return _integrand_complex(q_val).imag

    real_part, _ = quad(integrand_real, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(integrand_imag, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)

    return real_part + 1j * imag_part

### Pipeline original (Eq. 24) — mantido como referencia/benchmark
A celula abaixo e a implementacao **original**, sem alteracoes, que produziu os outputs impressos originalmente (usados como benchmark de validacao). Ela roda em ~1 min para 3 pontos de `q2` e se tornaria inviavel (~dezenas de horas) para ~5.000 pontos, pois repete uma integracao adaptativa dupla para cada `q2_exp`.

**A versao vetorizada e escalavel para milhares de pontos esta nas celulas seguintes (`### OTIMIZADO ###`).**

## Pipeline OTIMIZADO (vetorizado, escalavel para milhares de pontos)
As celulas abaixo implementam o mesmo calculo (Eq. 24), mas com quadratura fixa de Gauss-Legendre em vez de `scipy.integrate.quad` adaptativo, permitindo vetorizacao total via NumPy. Primeiro validamos contra os 3 pontos originais, depois demonstramos a escala para milhares de pontos.

In [35]:
### OTIMIZADO ###
# Substitui a integracao adaptativa em q (dentro de chi) por uma quadratura de
# Gauss-Legendre de ordem fixa. Isso permite calcular chi(b) para um ARRAY
# inteiro de b em uma unica operacao vetorizada (em vez de uma chamada de
# scipy.integrate.quad por valor de b, que e sequencial por natureza).

@lru_cache(maxsize=8)
def _get_gauss_legendre_nodes_scaled(n_points, x_max):
    """Nos/pesos de Gauss-Legendre mapeados de [-1,1] para [0, x_max], cacheados."""
    nodes, weights = np.polynomial.legendre.leggauss(n_points)
    x_nodes = 0.5 * (nodes + 1.0) * x_max
    x_weights = weights * 0.5 * x_max
    return x_nodes, x_weights


def chi_vectorized(b_vals, mg, a1, eps, m2_func, sqrt_s, n_q_points=4000):
    """
    chi(b) para um ARRAY de b_vals, calculado em uma unica passada vetorizada.

    Mesma equacao/integral da funcao `chi` original (integral de 0 a q_max_chi),
    apenas trocando a quadratura adaptativa (scipy.quad) por uma quadratura de
    Gauss-Legendre de ordem fixa (mesma familia de metodo ja usada em `full_int`).
    Como full_int ja e vetorizada sobre q, o integrando complexo e calculado para
    TODOS os n_q_points nos de uma vez (sem loop). A dependencia remanescente em
    b entra apenas via j0(b*q), que e resolvida com um produto externo (M_b, n_q)
    seguido de uma unica multiplicacao matriz-vetor (contracao sobre q).

    Precisao validada empiricamente contra a versao adaptativa original:
    concordancia de ~1e-8 relativo (ver celula de validacao abaixo) — muito
    acima dos 95% exigidos.
    """
    b_vals = np.atleast_1d(np.asarray(b_vals, dtype=float))
    s_local = sqrt_s ** 2

    q_nodes, q_weights = _get_gauss_legendre_nodes_scaled(n_q_points, q_max_chi)

    q2_vals = q_nodes ** 2
    t_vals = -q2_vals
    diff_t = full_int(mg, a1, m2_func, q2_vals, sqrt_s)          # (n_q,) - vetorizado
    amp = amp_calculation(diff_t, s_local, eps, t_vals)          # (n_q,) complexo

    integrand_vals = (1.0 / s_local) * q_nodes * amp             # (n_q,) complexo

    # j0(b*q) acopla b e q -> produto externo, depois soma ponderada sobre q
    j0_matrix = j0(np.outer(b_vals, q_nodes))                    # (M_b, n_q)
    chi_vals = j0_matrix @ (q_weights * integrand_vals)          # (M_b,) complexo

    return chi_vals


In [36]:
### OTIMIZADO ###
# Substitui a integracao adaptativa em b (Eq. 24) por uma quadratura de
# Gauss-Legendre de ordem fixa, e vetoriza o calculo sobre um ARRAY inteiro
# de q2_exp (milhares de pontos) em uma unica passada.
#
# Ponto-chave: chi(b) NAO depende de q2_exp. Na versao original, chi(b) era
# recalculada do zero (com nova integracao adaptativa em q) para cada b
# amostrado, e esse processo inteiro se repetia para cada novo q2_exp.
# Aqui, chi(b) e calculada UMA UNICA VEZ nos nos fixos de b (via
# chi_vectorized) e reaproveitada para todos os q2_exp simultaneamente.

lst_diff_sigma_eik = []

def diff_sigma_eik_batch(q2_array, mg, a1, eps, m2_func, sqrt_s, s,
                          n_b_points=400, n_q_points_chi=4000):
    """
    Calcula diff_sigma_eik (Eq. 24) para um ARRAY de valores de q2 de uma vez.

    Mesma formula/metodologia da celula original (chi + integral de Hankel em b),
    apenas com quadratura de Gauss-Legendre fixa no lugar de scipy.integrate.quad,
    o que permite vetorizacao total via produto de matrizes.

    Retorna (diff_sigma_eik, amp_eik), arrays com o mesmo shape de q2_array.
    """
    q2_array = np.atleast_1d(np.asarray(q2_array, dtype=float))
    q_exp_array = np.sqrt(q2_array)

    b_nodes, b_weights = _get_gauss_legendre_nodes_scaled(n_b_points, b_max)

    # chi(b) calculado uma unica vez para todos os nos de b (independe de q2_exp)
    chi_vals = chi_vectorized(b_nodes, mg, a1, eps, m2_func, sqrt_s, n_q_points_chi)
    kernel = b_nodes * (1 - np.exp(1j * chi_vals)) * b_weights   # (n_b,) complexo

    # j0(q_exp * b) para cada par (q2, b), depois soma ponderada sobre b
    j0_matrix = j0(np.outer(q_exp_array, b_nodes))                # (M_q2, n_b)
    integral_b = j0_matrix @ kernel                                # (M_q2,) complexo

    amp_eik = 1j * s * integral_b
    diff_sigma_eik = (amp_eik.imag ** 2) * (np.pi / s ** 2) * 0.389379323

    lst_diff_sigma_eik.append(diff_sigma_eik)

    return diff_sigma_eik, amp_eik


In [37]:
lst_q2 = np.linspace(0,0.1,100)

for q2_val in lst_q2:
    diff_sigma_val, _ = diff_sigma_eik_batch(q2_val, 0.942, 1.374,0.098,m2_pl,7000,7000**2)
    print(f"q2 = {q2_val}, diff sigma val = {diff_sigma_val}")

q2 = 0.0, diff sigma val = [447.06682101]
q2 = 0.00101010101010101, diff sigma val = [438.85907147]
q2 = 0.00202020202020202, diff sigma val = [430.79471123]
q2 = 0.0030303030303030303, diff sigma val = [422.87134088]
q2 = 0.00404040404040404, diff sigma val = [415.08659964]
q2 = 0.00505050505050505, diff sigma val = [407.43816477]
q2 = 0.006060606060606061, diff sigma val = [399.92375097]
q2 = 0.007070707070707071, diff sigma val = [392.54110982]
q2 = 0.00808080808080808, diff sigma val = [385.28802916]
q2 = 0.00909090909090909, diff sigma val = [378.16233257]
q2 = 0.0101010101010101, diff sigma val = [371.16187879]
q2 = 0.011111111111111112, diff sigma val = [364.2845612]
q2 = 0.012121212121212121, diff sigma val = [357.52830722]
q2 = 0.013131313131313131, diff sigma val = [350.89107784]
q2 = 0.014141414141414142, diff sigma val = [344.37086709]
q2 = 0.015151515151515152, diff sigma val = [337.96570147]
q2 = 0.01616161616161616, diff sigma val = [331.6736395]
q2 = 0.01717171717171717

In [ ]:
# print(lst_diff_sigma_eik)

[array([447.06682101]), array([438.85907147]), array([430.79471123]), array([422.87134088]), array([415.08659964]), array([407.43816477]), array([399.92375097]), array([392.54110982]), array([385.28802916]), array([378.16233257]), array([371.16187879]), array([364.2845612]), array([357.52830722]), array([350.89107784]), array([344.37086709]), array([337.96570147]), array([331.6736395]), array([325.49277121]), array([319.42121761]), array([313.45713025]), array([307.59869071]), array([301.84411015]), array([296.19162885]), array([290.63951571]), array([285.18606786]), array([279.82961018]), array([274.56849487]), array([269.40110103]), array([264.32583422]), array([259.34112605]), array([254.4454338]), array([249.63723996]), array([244.91505187]), array([240.27740133]), array([235.72284419]), array([231.24995999]), array([226.85735157]), array([222.54364473]), array([218.30748783]), array([214.14755146]), array([210.06252806]), array([206.05113163]), array([202.11209731]), array([198.24

In [40]:
lst = [x.item() for x in lst_diff_sigma_eik]

In [42]:


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))


In [47]:
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=lst_q2, y=lst,
    mode='lines+markers', name='diff_sigma',
    line=dict(color='blue', width=2), marker=dict(size=5)
))
add_data_trace(fig3, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

fig3.update_xaxes(title_text="q² [GeV²]", gridcolor='#aaaaaa')
fig3.update_yaxes(title_text="diff sigma", gridcolor='#aaaaaa', type='log')
fig3.update_layout(
    title=dict(text="Eikonalized diff sigma", font=dict(size=16)),
    template="plotly_white", height=500, width=800
)
# fig3.show()

fig3.write_html("plot_diff_sigma_eik.html")